# Brand Selection Analysis

This notebook analyzes the support conversation dataset to select the optimal brand for the assignment.

We evaluate each brand on:
- Tweet/message volume and conversation count
- Average conversation length and complexity
- Customer vs. agent message balance
- Data quality (duplicates, noise)
- Resolution identification capability
- Intent diversity


In [ ]:
import pandas as pd
import numpy as np
import json
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print("Libraries imported successfully.")

## 1. Load and Inspect Dataset

In [ ]:
# Load the dataset from sample.csv
try:
    df = pd.read_csv('../data/sample.csv')
    print("✓ Loaded sample.csv")
except FileNotFoundError:
    print("ERROR: sample.csv not found in data/ directory")
    df = None

if df is not None:
    print(f"\nDataset Shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nFirst few rows:")
    print(df.head(10))
    print(f"\nData types:\n{df.dtypes}")

## 2. Identify Conversation Structure

Map Twitter conversation threads using in_response_to_tweet_id and response_tweet_id

In [ ]:
# Build conversation threads
# A conversation is identified by finding all tweets that belong to the same thread

def build_conversation_id(df):
    """
    Map tweets to conversation IDs by following thread chains.
    """
    conversation_map = {}
    
    for idx, row in df.iterrows():
        tweet_id = row['tweet_id']
        in_response = row['in_response_to_tweet_id']
        
        # Start a new conversation or join existing
        if pd.isna(in_response):
            # This tweet starts a conversation
            conversation_map[tweet_id] = tweet_id
        else:
            # This tweet responds to another
            # Find the root tweet of the response chain
            root = in_response
            while root in df['in_response_to_tweet_id'].values:
                matching_row = df[df['tweet_id'] == root]
                if len(matching_row) > 0 and not pd.isna(matching_row.iloc[0]['in_response_to_tweet_id']):
                    root = matching_row.iloc[0]['in_response_to_tweet_id']
                else:
                    break
            conversation_map[tweet_id] = root
    
    # Map conversations to numeric IDs
    unique_convs = {v: i for i, v in enumerate(set(conversation_map.values()))}
    conversation_ids = [unique_convs[v] for v in conversation_map.values()]
    
    return conversation_ids

# Add conversation_id column
df['conversation_id'] = build_conversation_id(df)

print(f"Total unique conversations: {df['conversation_id'].nunique()}")
print(f"Total tweets: {len(df)}")
print(f"\nConversation ID mapping sample:")
print(df[['tweet_id', 'author_id', 'conversation_id']].head(20))

## 3. Brand Overview - Volume Analysis

In [ ]:
# Identify support brands (accounts that send responses)
# These are author_ids that have inbound=False (they are responding to customers)

support_brands = df[df['inbound'] == False]['author_id'].unique()
print(f"Support brand accounts detected: {len(support_brands)}")
print(f"Brands: {sorted(support_brands)}\n")

# Generate brand summary table
brand_summary = []

for brand in sorted(support_brands):
    # Get all messages (both inbound customer messages and outbound support responses) for this brand's conversation
    brand_convs = df[df['author_id'] == brand]['conversation_id'].unique()
    brand_data = df[df['conversation_id'].isin(brand_convs)]
    
    tweets = len(brand_data)
    conversations = len(brand_convs)
    avg_len = tweets / conversations if conversations > 0 else 0
    
    brand_summary.append({
        'Brand': brand,
        'Tweets': f"{tweets:,}",
        'Conversations': f"{conversations:,}",
        'Avg Msgs/Conv': f"{avg_len:.1f}"
    })

summary_df = pd.DataFrame(brand_summary)
print("BRAND OVERVIEW")
print("="*70)
print(summary_df.to_string(index=False))
print("="*70)

## 4. Detailed Analysis Per Brand

In [ ]:
# Detailed metrics for each brand
brand_metrics = {}

for brand in sorted(support_brands):
    brand_convs = df[df['author_id'] == brand]['conversation_id'].unique()
    brand_data = df[df['conversation_id'].isin(brand_convs)]
    
    # Basic counts
    num_tweets = len(brand_data)
    num_convs = len(brand_convs)
    
    # Conversation lengths
    conv_lengths = brand_data.groupby('conversation_id').size()
    avg_conv_len = conv_lengths.mean()
    max_conv_len = conv_lengths.max()
    min_conv_len = conv_lengths.min()
    
    # Multi-turn conversations (>=3 messages for meaningful dialogue)
    multi_turn = (conv_lengths >= 3).sum()
    multi_turn_pct = (multi_turn / num_convs * 100) if num_convs > 0 else 0
    
    # Duplicate analysis
    duplicates = brand_data.duplicated().sum()
    
    # Customer vs agent messages
    customer_msgs = (brand_data['inbound'] == True).sum()
    agent_msgs = (brand_data['inbound'] == False).sum()
    
    brand_metrics[brand] = {
        'tweets': num_tweets,
        'conversations': num_convs,
        'avg_conv_len': avg_conv_len,
        'max_conv_len': max_conv_len,
        'min_conv_len': min_conv_len,
        'multi_turn': multi_turn,
        'multi_turn_pct': multi_turn_pct,
        'duplicates': duplicates,
        'customer_msgs': customer_msgs,
        'agent_msgs': agent_msgs
    }

# Display detailed metrics
print("\nDETAILED BRAND METRICS")
print("="*90)
for brand, metrics in sorted(brand_metrics.items()):
    print(f"\n{brand.upper()}")
    print("-" * 90)
    print(f"  Total Tweets:              {metrics['tweets']:,}")
    print(f"  Total Conversations:       {metrics['conversations']:,}")
    print(f"  Avg Messages per Conv:     {metrics['avg_conv_len']:.2f}")
    print(f"  Conv Length Range:         {metrics['min_conv_len']} - {metrics['max_conv_len']}")
    print(f"  Multi-turn Conversations:  {metrics['multi_turn']:,} ({metrics['multi_turn_pct']:.1f}%)")
    print(f"  Duplicates:                {metrics['duplicates']:,}")
    print(f"  Customer Messages:         {metrics['customer_msgs']:,}")
    print(f"  Agent Messages:            {metrics['agent_msgs']:,}")

## 5. Resolution Detection Analysis

In [ ]:
# Identify resolutions by looking for common resolution keywords
resolution_keywords = [
    'resolved', 'fixed', 'solved', 'thank', 'thanks', 'appreciate',
    'helped', 'working', 'works', 'issue is closed', 'closed',
    'problem solved', 'answered', 'provided', 'solution',
    'support', 'assist', 'help', 'done', 'completed'
]

resolution_analysis = {}

for brand in sorted(support_brands):
    brand_convs = df[df['author_id'] == brand]['conversation_id'].unique()
    brand_data = df[df['conversation_id'].isin(brand_convs)]
    
    # Group by conversation
    conv_groups = brand_data.groupby('conversation_id')
    
    convs_with_resolution = 0
    
    for conv_id, conv_data in conv_groups:
        # Check if any message in the conversation contains resolution keywords
        conv_text = ' '.join(conv_data['text'].fillna('').astype(str)).lower()
        
        if any(kw in conv_text for kw in resolution_keywords):
            convs_with_resolution += 1
    
    total_convs = len(brand_convs)
    resolution_pct = (convs_with_resolution / total_convs * 100) if total_convs > 0 else 0
    
    resolution_analysis[brand] = {
        'convs_with_resolution': convs_with_resolution,
        'total_convs': total_convs,
        'resolution_pct': resolution_pct
    }

print("\nRESOLUTION DETECTION ANALYSIS")
print("="*70)
for brand, metrics in sorted(resolution_analysis.items()):
    print(f"\n{brand.upper()}")
    print(f"  Conversations with resolution indicators: {metrics['convs_with_resolution']:,} / {metrics['total_convs']:,}")
    print(f"  Resolution rate: {metrics['resolution_pct']:.1f}%")

## 6. Scoring and Recommendation

In [ ]:
# Calculate composite score for each brand
# Weights: conversation volume, multi-turn %, resolution rate, data quality

scores = {}

for brand in sorted(support_brands):
    metrics = brand_metrics[brand]
    
    # Normalize each metric to 0-100 scale
    max_convs = max([brand_metrics[b]['conversations'] for b in support_brands])
    conv_score = (metrics['conversations'] / max_convs) * 100 if max_convs > 0 else 0
    
    multi_turn_score = metrics['multi_turn_pct']  # Already 0-100
    avg_len_score = min(100, (metrics['avg_conv_len'] / 5) * 100)  # 5+ msgs is excellent
    resolution_score = resolution_analysis[brand]['resolution_pct']
    
    # Quality penalty for duplicates
    quality_penalty = min(10, metrics['duplicates'] / 10) if metrics['duplicates'] > 0 else 0
    
    # Composite score (weighted average)
    final_score = (
        conv_score * 0.30 +           # 30% conversation volume
        multi_turn_score * 0.25 +     # 25% multi-turn ratio
        avg_len_score * 0.20 +        # 20% conversation length
        resolution_score * 0.25 -     # 25% resolution rate
        quality_penalty                # Minus duplicates
    )
    
    scores[brand] = final_score

# Rank brands
ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

print("\nBRAND SCORING & RANKING")
print("="*70)
for rank, (brand, score) in enumerate(ranked, 1):
    print(f"{rank}. {brand:25s} → Score: {score:6.1f}")

# Select best brand
best_brand = ranked[0][0]
best_score = ranked[0][1]
best_metrics = brand_metrics[best_brand]

print(f"\n{'='*70}")
print(f"🎯 SELECTED BRAND: {best_brand.upper()}")
print(f"{'='*70}")

## 7. Final Recommendation & Justification

In [ ]:
# Generate final recommendation statement

best_resolution = resolution_analysis[best_brand]['resolution_pct']

recommendation = f"""
╔══════════════════════════════════════════════════════════════════════════╗
║                    BRAND SELECTION CONCLUSION                            ║
╚══════════════════════════════════════════════════════════════════════════╝

✅ SELECTED BRAND: {best_brand.upper()}

📊 KEY METRICS:
   • Total Conversations:            {best_metrics['conversations']:,}
   • Unique Messages/Tweets:         {best_metrics['tweets']:,}
   • Avg Messages per Conversation:  {best_metrics['avg_conv_len']:.2f}
   • Multi-turn Conversations:       {best_metrics['multi_turn']:,} ({best_metrics['multi_turn_pct']:.1f}%)
   • Conversations with Resolution:  {best_resolution:.1f}%
   • Data Quality (duplicates):      {best_metrics['duplicates']:,}

✓ JUSTIFICATION:

We selected {best_brand.upper()} because:

1. SUFFICIENT VOLUME:
   {best_metrics['conversations']:,} conversations provide sufficient data for training intent
   classifiers and building a 200-example gold set while maintaining
   reasonable diversity across support scenarios.

2. CONVERSATION COMPLEXITY:
   With an average of {best_metrics['avg_conv_len']:.2f} messages per conversation and
   {best_metrics['multi_turn_pct']:.1f}% multi-turn conversations (≥3 messages), this brand demonstrates
   adequate back-and-forth dialogue for developing retrieval-based
   response generation and meaningful agent escalation patterns.

3. RESOLVABLE PATTERNS:
   {best_resolution:.1f}% of conversations contain identifiable resolution indicators,
   enabling us to identify positive examples for the gold set and
   establish clear success criteria for the evaluation system.

4. INTENT DIVERSITY:
   The conversation volume and complexity suggest a rich set of
   recurring customer-support intents (e.g., account issues, billing,
   technical support, order status), suitable for building a
   meaningful intent taxonomy.

5. EVALUATION FEASIBILITY:
   This dataset is large enough for training robust baselines and
   evaluation while remaining manageable for a single-person assignment.

✓ READY FOR NEXT PHASE:
   Conversation reconstruction and intent taxonomy development can
   proceed with {best_brand.upper()}.

╚══════════════════════════════════════════════════════════════════════════╝
"""

print(recommendation)